# HW4 - FBS Team Ratings: SRS, Opponent-Adjusted Game Control, and Opponent-Adjusted AWP

This notebook builds three team-rating models for FBS football, 2021-2025, and evaluates them
two ways: predictive out-of-sample accuracy, and split-half repeatability.

**The three models** (Part 1, 30 pts):
1. **SRS** - opponent-adjusted rating built from final score margin, plus a home-field effect.
2. **Opponent-adjusted Game Control** - opponent-adjusted rating built from how large a scoring
   margin a team held throughout the game (not just the final score), plus a home-field effect.
3. **Opponent-adjusted AWP** - opponent-adjusted rating built from a team's average in-game win
   probability (using CFBD's own win-probability metric), plus a home-field effect.

All three share the same underlying regression machinery - they only differ in *what per-game number gets fed in* as the outcome being adjusted.
I did this on purpose which I explain in step 5.

**Evaluation** (Parts 2 and 3, 20 pts each): each season gets split two different ways - once
chronologically, and once randomly - with the reasoning for each split explained in its own section.


## Setup

1. `pip install pandas numpy requests statsmodels scipy python-dotenv` (statsmodels/scipy aren't
   strictly required by this notebook but are useful if you want to sanity-check results against
   `smf.ols`).
2. Get a free CFBD API key at https://collegefootballdata.com/key and put it in a `.env` file in
   this folder: `CFBD_API_KEY=your_key_here`.
3. Run All. Steps 1-2-4 fetch data from CFBD and cache it to CSV, same pattern as HW2, and fire
   several requests concurrently instead of one at a time since each request is mostly spent
   waiting on the network. **Step 4 (win probability) is still the slowest one** - CFBD only
   offers win probability one game at a time, and there are several thousand FBS games across
   five seasons. It writes its progress to CSV as it goes, so it's safe to stop and re-run - it
   will pick up where it left off instead of re-fetching games it already has.

None of the fetched data is committed to GitHub (see `.gitignore`) - only the code and this
notebook are pushed.


In [1]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import requests

HERE = Path.cwd()


def _load_dotenv(path=HERE / ".env"):
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())


_load_dotenv()


def get_api_key():
    key = os.environ.get("CFBD_API_KEY")
    if not key:
        raise RuntimeError(
            "CFBD_API_KEY not found. Put it in a .env file in this folder: CFBD_API_KEY=your_key_here"
        )
    return key


def cfbd_get(path, params=None, max_attempts=3):
    url = f"https://api.collegefootballdata.com{path}"
    headers = {"Authorization": f"Bearer {get_api_key()}"}
    for attempt in range(max_attempts):
        resp = requests.get(url, headers=headers, params=params or {})
        if resp.status_code == 429:
            time.sleep(2 * (attempt + 1))
            continue
        if resp.status_code != 200:
            raise RuntimeError(f"CFBD {path} {params} failed: {resp.status_code} {resp.text[:200]}")
        return resp.json()
    raise RuntimeError(f"CFBD {path} {params} failed after {max_attempts} attempts (rate limited)")


YEARS = [2021, 2022, 2023, 2024, 2025]
print(f"Seasons: {YEARS}")


Seasons: [2021, 2022, 2023, 2024, 2025]


## Step 1: FBS vs. FBS games, 2021-2025

Same idea as HW2's games fetch: pull every game for each season, keep only games where both
teams are FBS, and cache the result so this only ever hits the API once.


In [2]:
GAMES_PATH = HERE / "games_fbs.csv"

if GAMES_PATH.exists():
    games = pd.read_csv(GAMES_PATH)
    print(f"Loaded cached {GAMES_PATH.name}: {len(games)} games")
else:
    rows = []
    for year in YEARS:
        year_games = cfbd_get("/games", {"year": year, "seasonType": "regular"})
        for g in year_games:
            if g.get("homeClassification") != "fbs" or g.get("awayClassification") != "fbs":
                continue
            if g.get("homePoints") is None or g.get("awayPoints") is None:
                continue  # game not yet played / no final score
            rows.append({
                "game_id": g.get("id"),
                "season": year,
                "week": g.get("week"),
                "home_team": g.get("homeTeam"),
                "away_team": g.get("awayTeam"),
                "home_points": g.get("homePoints"),
                "away_points": g.get("awayPoints"),
            })
        print(f"{year}: done, {len(rows)} games so far")
    games = pd.DataFrame(rows)
    games.to_csv(GAMES_PATH, index=False)
    print(f"Wrote {GAMES_PATH.name}: {len(games)} games")

games["margin"] = games["home_points"] - games["away_points"]
print(f"Teams seen: {pd.concat([games['home_team'], games['away_team']]).nunique()}")
games.head()


Loaded cached games_fbs.csv: 3730 games
Teams seen: 136


,game_id,season,week,home_team,away_team,home_points,away_points,margin
0,401282714,2021,1,Illinois,Nebraska,30,22,8
1,401286187,2021,1,Fresno State,UConn,45,0,45
2,401309833,2021,1,UCLA,Hawai'i,44,10,34
3,401282049,2021,1,New Mexico State,UTEP,3,30,-27
4,401300995,2021,1,UCF,Boise State,36,31,5


## Step 2: play-by-play, for the Game Control metric

Pulled the same way as HW2 - per year/week, with the FBS-game filter applied at the play level
and a try/except so one bad week doesn't kill the whole fetch. The 75 year/week requests are fired concurrently rather than one at a time, since each is mostly network wait.


In [3]:
PLAYS_PATH = HERE / "plays_fbs.csv"

if PLAYS_PATH.exists():
    plays = pd.read_csv(PLAYS_PATH)
    print(f"Loaded cached {PLAYS_PATH.name}: {len(plays)} plays")
else:
    valid_game_ids = set(games["game_id"])
    keep_cols = ["game_id", "offense", "defense", "offense_score", "defense_score",
                 "home", "away", "play_type"]

    def fetch_week_plays(year_week):
        year, week = year_week
        try:
            week_plays = cfbd_get("/plays", {"year": year, "week": week, "seasonType": "regular"})
        except RuntimeError as e:
            print(f"  {year} week {week}: {e}")
            return []
        rows = []
        for p in week_plays:
            if p.get("gameId") not in valid_game_ids:
                continue
            rows.append({
                "game_id": p.get("gameId"),
                "offense": p.get("offense"),
                "defense": p.get("defense"),
                "offense_score": p.get("offenseScore"),
                "defense_score": p.get("defenseScore"),
                "home": p.get("homeTeam") if "homeTeam" in p else p.get("home"),
                "away": p.get("awayTeam") if "awayTeam" in p else p.get("away"),
                "play_type": p.get("playType"),
            })
        return rows

    # one call per (year, week) - only ~75 total, but fired concurrently instead of one at a
    # time since each call is almost entirely spent waiting on the network, not computing.
    year_weeks = [(year, week) for year in YEARS for week in range(1, 16)]
    all_rows = []
    with ThreadPoolExecutor(max_workers=15) as pool:
        futures = {pool.submit(fetch_week_plays, yw): yw for yw in year_weeks}
        done = 0
        for future in as_completed(futures):
            all_rows.extend(future.result())
            done += 1
            if done % 15 == 0 or done == len(year_weeks):
                print(f"  fetched {done} of {len(year_weeks)} year/week chunks ({len(all_rows)} plays so far)")

    plays = pd.DataFrame(all_rows)[keep_cols]
    plays.to_csv(PLAYS_PATH, index=False)
    print(f"Wrote {PLAYS_PATH.name}: {len(plays)} plays")

plays.head()


Loaded cached plays_fbs.csv: 664981 plays


,game_id,offense,defense,offense_score,defense_score,home,away,play_type
0,401281944,Auburn,Akron,0,0,Auburn,Akron,Kickoff
1,401281944,Auburn,Akron,0,0,Auburn,Akron,Kickoff
2,401281944,Akron,Auburn,0,0,Auburn,Akron,Timeout
3,401281944,Akron,Auburn,0,0,Auburn,Akron,Punt
4,401281944,Akron,Auburn,0,0,Auburn,Akron,Rush


## Step 3: build the Game Control metric

**Definition, and why:** SRS (Step 5) only looks at the final score - a team that led by 21
for three quarters and won by 3 after a garbage-time score looks identical to SRS as a team that
trailed the whole game and won 3-0 on a last-second field goal. Game Control is meant to capture
the difference: it's the average scoring margin a team held across every play in the game, not
just the final one.

I used a simple, unweighted average of the running margin at each real scrimmage play (excluding
kickoffs, timeouts, etc. - the same `KEEP_PLAY_TYPES` filtering idea from HW2), rather than
weighting by elapsed game time. 


In [4]:
KEEP_PLAY_TYPES = {
    "Rush", "Pass Reception", "Pass Incompletion", "Pass Completion", "Sack",
    "Rushing Touchdown", "Passing Touchdown", "Fumble Recovery (Own)",
    "Fumble Recovery (Opponent)", "Fumble", "Pass Interception Return",
    "Interception", "Interception Return Touchdown", "Fumble Return Touchdown",
    "Field Goal Good", "Field Goal Missed", "Blocked Field Goal", "Safety", "Pass",
}

gc = plays[plays["play_type"].isin(KEEP_PLAY_TYPES)].copy()

gc["home_score"] = np.where(gc["offense"] == gc["home"], gc["offense_score"], gc["defense_score"])
gc["away_score"] = np.where(gc["offense"] == gc["away"], gc["offense_score"], gc["defense_score"])
gc["running_margin"] = gc["home_score"] - gc["away_score"]

game_control = gc.groupby("game_id")["running_margin"].mean().rename("game_control_home").reset_index()

games = games.merge(game_control, on="game_id", how="left")
print(f"Games with a game-control value: {games['game_control_home'].notna().sum()} of {len(games)}")
games[["game_id", "home_team", "away_team", "margin", "game_control_home"]].head()


Games with a game-control value: 3728 of 3730


,game_id,home_team,away_team,margin,game_control_home
0,401282714,Illinois,Nebraska,8,5.460993
1,401286187,Fresno State,UConn,45,25.142857
2,401309833,UCLA,Hawai'i,34,25.078014
3,401282049,New Mexico State,UTEP,-27,-17.938356
4,401300995,UCF,Boise State,5,-5.649007


## Step 4: win probability, for the AWP metric

CFBD's `metrics/wp` endpoint (the same one used in HW3) returns play-by-play win probability,
but only one game at a time - there's no bulk endpoint for it. With several thousand FBS games
across five seasons, this is the slow step - fetching them one at a time, waiting for each
response before starting the next, would take a very long time.

Since each call is almost entirely spent waiting on the network (not computing), this fires
several requests at once with a thread pool (`ThreadPoolExecutor`) instead of one at a time.
Only the main thread ever writes to the cache file - each game's result gets **appended to the
cache as soon as it's fetched**, so if this gets interrupted (rate limit, network drop, you
closing your laptop), re-running the cell picks up only the games still missing instead of
starting over.


In [5]:
WP_PATH = HERE / "wp_by_game.csv"
MAX_WORKERS = 10  # a middle ground - 15 tripped the rate limiter, 6 was too conservative

if WP_PATH.exists():
    wp_cache = pd.read_csv(WP_PATH)
else:
    wp_cache = pd.DataFrame(columns=["game_id", "avg_wp_home"])
    wp_cache.to_csv(WP_PATH, index=False)

already_done = set(wp_cache["game_id"])
remaining = [gid for gid in games["game_id"] if gid not in already_done]
print(f"{len(already_done)} games already cached, {len(remaining)} left to fetch")


def fetch_wp(game_id):
    """Runs on a worker thread - only fetches and computes, never touches the CSV."""
    try:
        wp_plays = cfbd_get("/metrics/wp", {"gameId": game_id})
    except RuntimeError as e:
        return game_id, None, str(e)
    if not wp_plays:
        return game_id, None, None
    avg_wp_home = float(np.mean([p.get("homeWinProbability") for p in wp_plays
                                  if p.get("homeWinProbability") is not None]))
    return game_id, avg_wp_home, None


fetched, errors = 0, 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(fetch_wp, gid): gid for gid in remaining}
    for future in as_completed(futures):
        game_id, avg_wp_home, err = future.result()
        if err is not None:
            errors += 1
            continue
        if avg_wp_home is not None:
            row = pd.DataFrame([{"game_id": game_id, "avg_wp_home": avg_wp_home}])
            row.to_csv(WP_PATH, mode="a", header=False, index=False)
        fetched += 1
        if (fetched + errors) % 50 == 0:
            print(f"  {fetched + errors} of {len(remaining)} done ({fetched} fetched, {errors} rate-limited)")

if errors:
    print(f"{errors} games were rate-limited - just re-run this cell, it only retries what's missing "
          f"(usually clears up in 1-2 more runs)")

wp_cache = pd.read_csv(WP_PATH)
games = games.merge(wp_cache, on="game_id", how="left")
print(f"Games with an AWP value: {games['avg_wp_home'].notna().sum()} of {len(games)}")

2903 games already cached, 827 left to fetch
827 games were rate-limited - just re-run this cell, it only retries what's missing (usually clears up in 1-2 more runs)
Games with an AWP value: 2903 of 3730


## Step 5: the shared opponent-adjustment regression

All three models are actually the same regression - the only thing that changes is what number I'm predicting (margin, game control, or win probability). For every game, I get one row of data that looks like:

y_game = HFA + rating[home_team] - rating[away_team] + error

I fit this with ordinary least squares because one column per team (+1 if they're home, -1 if they're away, 0 if they're not in that game) plus one extra column for home-field advantage. It's just the standard "SRS via regression" trick, except now it works for any per-game number, not just margin.

One thing I noticed was that you can shift every team's rating up by the same amount and nothing actually changes, since it's really the difference between two teams' ratings that matters, not their absolute values. That means the math technically has infinite solutions. 

np.linalg.lstsq handles this by picking the smallest possible solution, which happens to land on ratings that average out to zero. 

For AWP, I'm treating a probability (something between 0 and 1) as a regular number I can run linear regression on. Since average win probability over a whole game rarely gets close to 0 or 1, and I just care about ranking teams relative to each other rather than predicting the probability itself, plain OLS here is a reasonable shortcut - same idea as SRS just being linear regression on margin.

In [6]:
def fit_team_ratings(games_df, response_col, team_list):
    """
    Opponent-adjusted rating via linear regression with team fixed effects + home-field
    advantage. response = HFA + rating[home_team] - rating[away_team] + error.
    Returns (ratings: Series indexed by team, hfa: float).
    """
    team_index = {t: i for i, t in enumerate(team_list)}
    n_games, n_teams = len(games_df), len(team_list)
    X = np.zeros((n_games, n_teams + 1))
    y = games_df[response_col].to_numpy()

    home_idx = games_df["home_team"].map(team_index).to_numpy()
    away_idx = games_df["away_team"].map(team_index).to_numpy()
    X[np.arange(n_games), home_idx] = 1.0
    X[np.arange(n_games), away_idx] = -1.0
    X[:, n_teams] = 1.0  # HFA

    coefs, *_ = np.linalg.lstsq(X, y, rcond=None)
    ratings = pd.Series(coefs[:n_teams], index=team_list, name=response_col)
    hfa = coefs[n_teams]
    return ratings, hfa


MODELS = {
    "SRS": "margin",
    "Game Control": "game_control_home",
    "AWP": "avg_wp_home",
}

season_ratings = {}  # season_ratings[season][model_name] = (ratings, hfa)

for season in YEARS:
    season_games = games[games["season"] == season].dropna(
        subset=["margin", "game_control_home", "avg_wp_home"]
    )
    team_list = sorted(pd.concat([season_games["home_team"], season_games["away_team"]]).unique())
    season_ratings[season] = {}
    for model_name, response_col in MODELS.items():
        ratings, hfa = fit_team_ratings(season_games, response_col, team_list)
        season_ratings[season][model_name] = (ratings, hfa)
    print(f"{season}: fit all 3 models on {len(season_games)} games, {len(team_list)} teams")


2021: fit all 3 models on 699 games, 130 teams
2022: fit all 3 models on 600 games, 131 teams
2023: fit all 3 models on 624 games, 133 teams
2024: fit all 3 models on 615 games, 134 teams
2025: fit all 3 models on 365 games, 136 teams


## Step 6: top 10 / bottom 10 teams, per model, per season


In [7]:
for season in YEARS:
    for model_name in MODELS:
        ratings, hfa = season_ratings[season][model_name]
        ranked = ratings.sort_values(ascending=False)
        print(f"\n=== {season} - {model_name} (HFA = {hfa:.3f}) ===")
        print("Top 10:")
        print(ranked.head(10).to_string())
        print("Bottom 10:")
        print(ranked.tail(10).to_string())



=== 2021 - SRS (HFA = 1.673) ===
Top 10:
Georgia           34.547115
Ohio State        29.965083
Alabama           29.764566
Michigan          24.269284
Notre Dame        21.607093
Utah              19.234894
Cincinnati        18.477656
Oklahoma State    18.048250
Iowa State        17.116281
Wisconsin         16.505826
Bottom 10:
Rice                    -20.328622
Kansas                  -20.376937
Arkansas State          -21.652040
New Mexico State        -21.715103
Southern Miss           -25.347198
UConn                   -28.334638
Temple                  -28.994980
Akron                   -29.872475
Massachusetts           -30.658916
Florida International   -33.956837

=== 2021 - Game Control (HFA = 1.414) ===
Top 10:
Georgia       21.056044
Alabama       18.398875
Ohio State    16.027496
Michigan      12.689074
Ole Miss      10.805576
Cincinnati    10.694402
Tennessee     10.534256
Utah          10.528033
Notre Dame     9.794998
Wisconsin      9.736599
Bottom 10:
New Mexico Stat

## Step 7: predictive out-of-sample performance


In [8]:
TRAIN_FRACTION = 0.7
predictive_results = []

for season in YEARS:
    season_games = games[games["season"] == season].dropna(
        subset=["margin", "game_control_home", "avg_wp_home", "week"]
    )
    cutoff_week = season_games["week"].quantile(TRAIN_FRACTION)
    train = season_games[season_games["week"] <= cutoff_week]
    test = season_games[season_games["week"] > cutoff_week]
    if len(test) == 0:
        continue
    team_list = sorted(pd.concat([train["home_team"], train["away_team"]]).unique())

    for model_name, response_col in MODELS.items():
        ratings, hfa = fit_team_ratings(train, response_col, team_list)

        test_known = test[test["home_team"].isin(team_list) & test["away_team"].isin(team_list)]
        pred = test_known["home_team"].map(ratings) - test_known["away_team"].map(ratings) + hfa
        actual = test_known["margin"]  # always evaluate against real point margin / real winner
        rmse = float(np.sqrt(np.mean((pred - actual) ** 2)))
        accuracy = float(((pred > 0) == (actual > 0)).mean())

        predictive_results.append({
            "season": season, "model": model_name,
            "train_games": len(train), "test_games": len(test_known),
            "test_rmse_margin": rmse, "test_winner_accuracy": accuracy,
        })

predictive_df = pd.DataFrame(predictive_results)
predictive_df


,season,model,train_games,test_games,test_rmse_margin,test_winner_accuracy
0,2021,SRS,533,166,17.304816,0.740964
1,2021,Game Control,533,166,18.251548,0.759036
2,2021,AWP,533,166,22.144694,0.596386
3,2022,SRS,436,164,18.076722,0.682927
4,2022,Game Control,436,164,17.259110,0.713415
5,2022,AWP,436,164,19.513445,0.536585
6,2023,SRS,455,169,17.631364,0.751479
7,2023,Game Control,455,169,17.448837,0.775148
8,2023,AWP,455,169,20.393735,0.556213
9,2024,SRS,460,155,18.115008,0.638710


In [9]:
predictive_summary = predictive_df.groupby("model")[["test_rmse_margin", "test_winner_accuracy"]].mean()
print("Averaged across all 5 seasons:")
predictive_summary


Averaged across all 5 seasons:


,test_rmse_margin,test_winner_accuracy
model,,
AWP,20.007264,0.573438
Game Control,17.146278,0.716880
SRS,17.633901,0.710543


**Predictive out-of-sample performance. My Problem 2 answer(20 points)** 

I split each season chronologically and trained on the first 70% of the weeks, then tested on the rest. I decided to do this because it's  how you'd actually use this: you only ever have past games to work with when you're trying to predict future ones. If I'd split randomly instead, the model would get to peek at games that happened after the ones it's being tested on, which isn't a fair test of how well it actually predicts.

Game Control came out on top — lowest average RMSE (17.15) and best win accuracy (71.7%) across the five seasons. SRS was right behind it (17.63 RMSE, 71.1% accuracy), close enough that SRS actually wins some individual seasons (2021 on RMSE, 2024 and 2025 on accuracy), but Game Control wins more often overall so I'm calling it the better predictor. AWP was just worse across the board — worst RMSE and worst accuracy in every single season, averaging 20.0 RMSE and only 57.3% accuracy, barely above a coin flip some years. Both metrics agree with each other here, which makes me more confident in the answer: 

score-margin-based models beat AWP at predicting future games, and factoring in how much a team controlled the game (not just the final score) gives Game Control a small edge over plain SRS.


## Step 8: repeatability (Part 3, 20 pts)

Split choice: random, game-by-game, within each season - not chronological.

In [10]:
RNG = np.random.default_rng(2025)
repeatability_results = []

for season in YEARS:
    season_games = games[games["season"] == season].dropna(
        subset=["margin", "game_control_home", "avg_wp_home"]
    ).reset_index(drop=True)
    team_list = sorted(pd.concat([season_games["home_team"], season_games["away_team"]]).unique())

    shuffled_idx = RNG.permutation(season_games.index)
    half_a_idx, half_b_idx = np.array_split(shuffled_idx, 2)
    half_a = season_games.loc[half_a_idx]
    half_b = season_games.loc[half_b_idx]

    for model_name, response_col in MODELS.items():
        ratings_a, _ = fit_team_ratings(half_a, response_col, team_list)
        ratings_b, _ = fit_team_ratings(half_b, response_col, team_list)
        corr = float(np.corrcoef(ratings_a.reindex(team_list), ratings_b.reindex(team_list))[0, 1])
        repeatability_results.append({
            "season": season, "model": model_name,
            "half_a_games": len(half_a), "half_b_games": len(half_b),
            "split_half_correlation": corr,
        })

repeatability_df = pd.DataFrame(repeatability_results)
repeatability_df


,season,model,half_a_games,half_b_games,split_half_correlation
0,2021,SRS,350,349,0.795759
1,2021,Game Control,350,349,0.708761
2,2021,AWP,350,349,0.626408
3,2022,SRS,300,300,0.640406
4,2022,Game Control,300,300,0.608095
5,2022,AWP,300,300,0.562951
6,2023,SRS,312,312,0.592796
7,2023,Game Control,312,312,0.539451
8,2023,AWP,312,312,0.509947
9,2024,SRS,308,307,0.669490


In [11]:
repeatability_summary = repeatability_df.groupby("model")["split_half_correlation"].mean()
print("Averaged across all 5 seasons:")
repeatability_summary


Averaged across all 5 seasons:


model
AWP             0.499662
Game Control    0.567068
SRS             0.617220
Name: split_half_correlation, dtype: float64

**Part 3 — Repeatability(Question 3, 20 points)** For this one I split each season randomly, game by game, into two halves. Repeatability isn't about predicting anything, it's about whether the rating itself is stable. Since it's not about forecasting, there's no reason to care about time order here, so a random split makes more sense.

SRS won this one — highest average split-half correlation (0.617), and it wins 4 out of 5 seasons. Game Control was second (0.567 average) and actually beat SRS in 2025 (0.409 vs 0.388), but lost every other season. AWP was the least repeatable every single season, averaging just 0.500. 

What I found interesting is this is a different winner than Part 2 — Game Control predicted better, but SRS is more consistent as a measurement. Not really a contradiction though because SRS is simpler and just measures the same thing more reliably, while Game Control picks up more real signal about how a team played, which makes it a little noisier game to game. AWP is worse on both counts, so averaging win probability across a game just doesn't capture team strength as cleanly as scoring margin does. One thing I noticed was that 2025 has noticeably lower correlations across all three models (SRS drops all the way to 0.39, its worst season by far), and that lines up with 2025 having way fewer total games in the dataset than the other seasons (~365 vs 600-700). Smaller sample means a noisier split, so I wouldn't read too much into that season on its own.
